# 05 — Commercial Planning, Variance and Forecast

This notebook is the analytical trail behind dashboard pages 8 and 9 and the
Excel management pack. It picks up where
[`01_data_quality_and_core_analysis`](01_data_quality_and_core_analysis.ipynb)
stops: that notebook establishes *what happened*, this one establishes *why the
number moved* and *where it is heading*.

**Two things stated up front.** TheLook is synthetic, so the patterns come from a
generator rather than a market. And the plan data is constructed by
`src/build_plan_source.py` from a documented rule, because the dataset ships no
budget — it is a planning *scenario*, not source data. Both are restated wherever
a number depends on them.

Everything below is generated from the current build, so it cannot drift from the
dashboard or from `docs/commercial_analytics.md`.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
connection = duckdb.connect(
    str(PROJECT_ROOT / "artifacts" / "thelook_analytics.duckdb"),
    read_only=True,
)
connection.execute("SELECT 1").fetchone()

(1,)

## 1. The second source, and the master data it exposed

Plan data never arrives from the transactional system. It comes from regional
teams in spreadsheets, in local currency. Joining it to actuals is where master
data stops being a theoretical concern.

The mapping below is not invented for the exercise: TheLook's own `users.csv`
carries `España` next to `Spain`, `Deutschland` next to `Germany`, and `Brasil`
for Brazil.

In [2]:
# Every source spelling, and what it maps to.
connection.execute("""
    SELECT source_country_value, canonical_country, region, is_alias, alias_reason
    FROM core.dim_country
    ORDER BY is_alias DESC, canonical_country
""").fetchdf()

,source_country_value,canonical_country,region,is_alias,alias_reason
0,Brasil,Brazil,AMER,1,Portuguese-language spelling emitted by the so...
1,Deutschland,Germany,EMEA,1,Local-language spelling of Germany found in so...
2,España,Spain,EMEA,1,Local-language spelling of Spain found in sour...
3,Australia,Australia,APAC,0,NaN
4,Belgium,Belgium,EMEA,0,NaN
5,China,China,APAC,0,NaN
6,Colombia,Colombia,AMER,0,NaN
7,France,France,EMEA,0,NaN
8,Germany,Germany,EMEA,0,NaN
9,Japan,Japan,APAC,0,NaN


Unmapped, these three spellings would appear as separate markets. Two of them
carry real revenue that belongs to a market reported elsewhere, and one is a
near-zero row that would still pollute any country count.

In [3]:
# What the aliases would have cost if left unmapped: three phantom markets,
# two of them carrying real revenue away from the market they belong to.
connection.execute("""
    SELECT
        c.country          AS source_spelling,
        d.canonical_country,
        COUNT(*)           AS items,
        ROUND(SUM(f.net_sales_value), 2) AS net_sales_usd
    FROM core.fact_order_item f
    JOIN core.dim_customer c ON f.customer_id = c.user_id
    JOIN (SELECT DISTINCT source_country_value, canonical_country, is_alias
          FROM core.dim_country) d
      ON c.country = d.source_country_value
    WHERE d.is_alias = 1
    GROUP BY 1, 2
    ORDER BY 4 DESC
""").fetchdf()

,source_spelling,canonical_country,items,net_sales_usd
0,Brasil,Brazil,26098,1170130.54
1,Deutschland,Germany,1,14.95
2,España,Spain,1,0.00


## 2. Validation: what blocks, and what is quarantined

The design decision here matters more than the checks themselves.

**Structural defects block the build** — an unmapped country, a broken grain, an
FX conversion that does not reproduce, a consolidation that does not tie back to
the submissions, a bridge whose effects do not sum. Each of these makes the
consolidated number *wrong*, and publishing a wrong number is worse than not
publishing.

**Submission defects do not block** — blank cells, a sign error, a late region.
These make the plan *incomplete* rather than wrong. The month-end review still has
to happen; the defect goes back to the region that submitted it.

In [4]:
connection.execute("""
    SELECT check_name, severity, failure_count, denominator, status, notes
    FROM qa.plan_validation
    ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'WARN' THEN 2 ELSE 3 END, check_name
""").fetchdf()

,check_name,severity,failure_count,denominator,status,notes
0,plan_country_key_clean,MEDIUM,7,292,WARN,Whitespace in a join key silently creates an u...
1,plan_month_coverage_contiguous,HIGH,24,287,WARN,A gap means a region submitted late or skipped...
2,plan_target_not_blank,HIGH,4,292,WARN,Blank plan cells are quarantined and returned ...
3,plan_target_not_negative,HIGH,1,292,WARN,Negative targets indicate a sign error in the ...
4,actuals_tie_to_fact_order_item,CRITICAL,0,1,PASS,Country-grain actuals reconcile to the item fa...
5,plan_consolidation_ties_to_submissions,CRITICAL,0,1,PASS,Consolidated plan equals the sum of accepted s...
6,plan_country_resolves_to_master,CRITICAL,0,292,PASS,Every submitted country spelling maps to a can...
7,plan_currency_has_budget_rate,CRITICAL,0,292,PASS,Every submitted currency has a budget FX rate
8,plan_fx_conversion_reconciles,CRITICAL,0,292,PASS,Local-to-USD conversion reproduces exactly at ...
9,plan_grain_unique_country_month,CRITICAL,0,287,PASS,fact_plan holds one row per country per month


In [5]:
connection.execute("""
    SELECT reject_reason, COUNT(*) AS rows_quarantined,
           STRING_AGG(DISTINCT submitted_country, ', ') AS markets
    FROM qa.plan_rejects
    GROUP BY 1
""").fetchdf()

,reject_reason,rows_quarantined,markets
0,Plan value is blank,4,"United Kingdom, France, Japan, Australia"
1,Plan value is negative,1,Belgium


### One check, three root causes

The coverage check looks for gaps in a market's own plan months. It is worth
reading the result rather than just its pass/fail status, because the gaps come
from three unrelated causes:

- **Colombia** — the market is too small to plan. Sixteen orders in total, so most
  months have no prior-year base to plan from. A genuine business condition.
- **Poland** — a late submission. Two months never arrived.
- **Single-month gaps elsewhere** — the quarantined rows, showing up downstream.

A rule written against the *shape* of the data catches causes nobody listed in
advance. A rule written against a list of known problems does not.

In [6]:
# The coverage check reports gaps. Three different causes sit behind them,
# which is the argument for testing the shape of the data rather than a list
# of problems already known about.
connection.execute("""
    WITH bounds AS (
        SELECT canonical_country, MIN(plan_month) AS first_month, MAX(plan_month) AS last_month
        FROM core.fact_plan GROUP BY 1
    ),
    expected AS (
        SELECT b.canonical_country, CAST(m.value AS DATE) AS plan_month
        FROM bounds b, generate_series(b.first_month, b.last_month, INTERVAL 1 MONTH) AS m(value)
    )
    SELECT e.canonical_country, COUNT(*) AS missing_months
    FROM expected e
    LEFT JOIN core.fact_plan p
      ON e.canonical_country = p.canonical_country AND e.plan_month = p.plan_month
    WHERE p.canonical_country IS NULL
    GROUP BY 1 ORDER BY 2 DESC
""").fetchdf()

,canonical_country,missing_months
0,Colombia,16
1,Poland,3
2,Belgium,1
3,United Kingdom,1
4,Japan,1
5,Australia,1
6,France,1


## 3. Reconciliation

The statement every reviewer should be handed before they are asked to trust a
dashboard: what entered, what survived each layer, and where any difference comes
from. The sales stream must carry row count and value through unchanged; the plan
stream must lose exactly the quarantined rows and nothing else.

In [7]:
connection.execute("""
    SELECT stream, stage, entity, row_count, ROUND(value_usd, 2) AS value_usd,
           row_delta, expectation
    FROM qa.reconciliation_statement
    ORDER BY stream, step_order
""").fetchdf()

,stream,stage,entity,row_count,value_usd,row_delta,expectation
0,Plan,Source file,regional_plan_submissions.csv,292,NaN,<NA>,Submitted in local currency - no USD value yet
1,Plan,stg,stg.plan_submissions,292,8078399.98,0,Harmonised and FX-converted - row count must e...
2,Plan,qa,qa.plan_rejects,5,NaN,-287,"Quarantined rows, reported back to the submitt..."
3,Plan,core,core.fact_plan,287,8081299.98,282,Accepted rows only - source rows minus rejects
4,Sales,Source file,order_items.csv,180862,10796335.70,<NA>,Baseline: rows and gross value as delivered
5,Sales,stg,stg.order_items,180862,10796335.70,0,Typing and renaming only - must equal source
6,Sales,core,core.fact_order_item,180862,10796335.70,0,Joins to product and customer - must not drop ...
7,Sales,mart,mart.actual_country_month,135749,10796335.70,-45113,Aggregated to country x month - gross value mu...


## 4. Price-Volume-Mix

This closes a question left open in
[`analysis_and_findings.md` section 2](../docs/analysis_and_findings.md):
*is growth driven by volume or basket value?* The descriptive layer could show
revenue rising; it could not attribute the movement.

```
volume = (Q1 - Q0) x s0 x P0      growth held at prior mix and prior price
mix    = Q1 x (s1 - s0) x P0      mix shift held at prior price
price  = Q1_i x (P1 - P0)         price movement at current volume
```

Q is units, P is average selling price, s is the category's share of total units.

In [8]:
bridge = connection.execute("""
    SELECT comparison_label,
           ROUND(SUM(revenue_prior), 0)   AS revenue_prior,
           ROUND(SUM(revenue_current), 0) AS revenue_current,
           ROUND(SUM(total_variance), 0)  AS total_variance,
           ROUND(SUM(volume_effect), 0)   AS volume,
           ROUND(SUM(mix_effect), 0)      AS mix,
           ROUND(SUM(price_effect), 0)    AS price,
           ROUND(SUM(volume_effect) + SUM(mix_effect) + SUM(price_effect)
                 - SUM(total_variance), 6) AS residual
    FROM mart.price_volume_mix
    GROUP BY 1, comparison_order ORDER BY comparison_order
""").fetchdf()
bridge

,comparison_label,revenue_prior,revenue_current,total_variance,volume,mix,price,residual
0,CY2023 vs CY2022,1583799.0,2793942.0,1210144.0,1229590.0,-9499.0,-9948.0,-0.0
1,YTD Jan-Apr 2024 vs 2023,734018.0,1722202.0,988185.0,972255.0,-6177.0,22107.0,0.0
2,Apr 2024 vs Apr 2023,193150.0,550158.0,357008.0,349337.0,-5773.0,13444.0,0.0


In [9]:
# The decomposition must be exact. An off-by-rounding bridge is the classic
# way this analysis quietly breaks, so it is asserted rather than eyeballed.
assert bridge["residual"].abs().max() < 0.01, bridge[["comparison_label", "residual"]]
print(f"Bridge ties on all {len(bridge)} comparisons. "
      f"Largest residual: {bridge['residual'].abs().max():.6f} USD")

Bridge ties on all 3 comparisons. Largest residual: 0.000000 USD


### The finding

Average selling price barely moves, and volume accounts for almost the whole
variance in every comparison.

In [10]:
connection.execute("""
    SELECT comparison_label,
           ROUND(SUM(revenue_prior) / SUM(units_prior), 2)     AS asp_prior,
           ROUND(SUM(revenue_current) / SUM(units_current), 2) AS asp_current,
           ROUND(SUM(revenue_current) / SUM(units_current)
                 / (SUM(revenue_prior) / SUM(units_prior)) - 1, 4) AS asp_change,
           ROUND(SUM(volume_effect) / SUM(total_variance), 4)  AS volume_share_of_growth
    FROM mart.price_volume_mix
    GROUP BY 1, comparison_order ORDER BY comparison_order
""").fetchdf()

,comparison_label,asp_prior,asp_current,asp_change,volume_share_of_growth
0,CY2023 vs CY2022,59.82,59.41,-0.0069,1.0161
1,YTD Jan-Apr 2024 vs 2023,59.59,60.15,0.0093,0.9839
2,Apr 2024 vs Apr 2023,59.52,60.36,0.0141,0.9785


**This business grows by selling more units, not by charging more and not by
shifting into premium categories.** Two consequences follow. Growth is fully
exposed to traffic and conversion, with nothing cushioning a slowdown. And price
and mix are untouched levers — a one percent ASP improvement on current volume is
worth roughly $30K a quarter on its own.

The category view shows the same pattern repeating rather than one category
driving it.

In [11]:
connection.execute("""
    SELECT department, category,
           ROUND(total_variance, 0) AS total_variance,
           ROUND(volume_effect, 0)  AS volume,
           ROUND(mix_effect, 0)     AS mix,
           ROUND(price_effect, 0)   AS price
    FROM mart.price_volume_mix
    WHERE comparison_label LIKE 'YTD%'
    ORDER BY total_variance DESC
    LIMIT 10
""").fetchdf()

,department,category,total_variance,volume,mix,price
0,Men,Outerwear & Coats,81811.0,78907.0,2242.0,661.0
1,Men,Jeans,73195.0,69400.0,916.0,2879.0
2,Men,Suits & Sport Coats,68878.0,51209.0,12827.0,4842.0
3,Men,Sweaters,46872.0,52203.0,-4815.0,-516.0
4,Women,Intimates,44706.0,39161.0,3366.0,2179.0
5,Women,Dresses,44666.0,38857.0,-1347.0,7156.0
6,Men,Pants,39606.0,39159.0,-522.0,968.0
7,Women,Outerwear & Coats,38125.0,42067.0,-3899.0,-42.0
8,Women,Jeans,37141.0,46331.0,-9972.0,782.0
9,Men,Sleep & Lounge,33176.0,30326.0,3160.0,-310.0


## 5. Plan versus actual

Read 2023 and 2024 differently. 2023 landed close to plan in all three regions,
which is the evidence that the variance method works. 2024 runs far ahead
everywhere because the plan was built on the prior year's growth rate and the
generator's growth accelerated instead — that is an artifact of synthetic data,
not commercial outperformance, and it should not be presented as a win.

In [12]:
connection.execute("""
    SELECT region, EXTRACT(year FROM plan_year)::INTEGER AS plan_year,
           ROUND(actual_net_sales_usd, 0) AS actual_usd,
           ROUND(plan_net_sales_usd, 0)   AS plan_usd,
           ROUND(achievement_rate, 3)     AS achievement,
           months_without_plan
    FROM mart.plan_vs_actual_region
    ORDER BY plan_year, region
""").fetchdf()

,region,plan_year,actual_usd,plan_usd,achievement,months_without_plan
0,AMER,2023,1024574.0,1046200.0,0.979,3.0
1,APAC,2023,1226648.0,1301700.0,0.942,1.0
2,EMEA,2023,542721.0,512400.0,1.059,2.0
3,AMER,2024,636848.0,503600.0,1.265,2.0
4,APAC,2024,755784.0,618600.0,1.222,1.0
5,EMEA,2024,329570.0,234600.0,1.405,2.0


Only two markets land below plan across complete months. They are the same two the
coverage check flagged, which is corroboration rather than coincidence: a market
too small to plan reliably is also a market likely to miss.

In [13]:
connection.execute("""
    SELECT canonical_country, region,
           ROUND(SUM(actual_net_sales_usd), 0) AS actual_usd,
           ROUND(SUM(plan_net_sales_usd), 0)   AS plan_usd,
           ROUND(SUM(actual_net_sales_usd) / SUM(plan_net_sales_usd), 3) AS achievement
    FROM mart.plan_vs_actual_monthly
    WHERE is_complete_month = 1 AND in_plan_horizon = 1
      AND plan_net_sales_usd IS NOT NULL AND actual_net_sales_usd IS NOT NULL
    GROUP BY 1, 2 ORDER BY achievement
""").fetchdf()

,canonical_country,region,actual_usd,plan_usd,achievement
0,Poland,EMEA,8357.0,11600.0,0.720
1,Colombia,AMER,85.0,100.0,0.850
2,China,APAC,1527852.0,1506800.0,1.014
3,Japan,APAC,102493.0,99200.0,1.033
4,South Korea,APAC,244565.0,235900.0,1.037
5,United States,AMER,1012384.0,961500.0,1.053
6,Belgium,EMEA,52944.0,50100.0,1.057
7,Brazil,AMER,648034.0,588200.0,1.102
8,Australia,APAC,88594.0,78400.0,1.130
9,Germany,EMEA,197974.0,174300.0,1.136


## 6. Forecast, validated before it is used

Three conventional candidates, selected on **backtested** accuracy rather than
in-sample fit. Rolling origin, one month ahead, refitting at each origin so no
future information reaches the fit.

The weak benchmark is published deliberately: seasonal naive at ~54% MAPE is what
shows why the drift term matters on a series growing this fast, and it is the bar
any method had to clear before it earned its complexity.

In [14]:
connection.execute("""
    SELECT model, ROUND(mape, 4) AS mape,
           ROUND(worst_month_ape, 4) AS worst_month_ape,
           ROUND(bias, 4) AS mean_bias, folds
    FROM mart.forecast_accuracy ORDER BY mape
""").fetchdf()

,model,mape,worst_month_ape,mean_bias,folds
0,seasonal_naive_drift,0.0987,0.1899,-0.0659,6
1,holt_winters_log,0.1335,0.2489,0.0061,6
2,seasonal_naive,0.5396,0.6489,-0.5396,6


**Two things worth stating rather than burying.** The simplest trend-aware method
beat Holt-Winters, which means the series does not have structure that more
machinery would exploit. And the lowest-MAPE model is biased several points *low*
while Holt-Winters is nearly unbiased — for a planning committee that cares more
about not under-committing than about average error, the runner-up may be the
better choice. The table is published so that trade-off is visible.

**Honest limitation.** This MAPE is optimistic. Synthetic growth is far smoother
than real retail demand, which carries promotional spikes, stockouts and
competitive shocks. Read the number as a demonstration of the validation method,
not as an accuracy claim that transfers.

In [15]:
connection.execute("""
    SELECT month_start, basis,
           ROUND(expected_net_sales_usd, 0) AS expected_usd,
           ROUND(plan_net_sales_usd, 0)     AS plan_usd,
           ROUND(gap_to_plan_usd, 0)        AS gap_usd,
           ROUND(expected_achievement_rate, 3) AS expected_achievement
    FROM mart.forecast_vs_plan
    WHERE plan_net_sales_usd IS NOT NULL
    ORDER BY month_start
""").fetchdf()

,month_start,basis,expected_usd,plan_usd,gap_usd,expected_achievement
0,2023-01-01,Actual,188680.0,204000.0,-15320.0,0.925
1,2023-02-01,Actual,167410.0,177000.0,-9590.0,0.946
2,2023-03-01,Actual,184778.0,214900.0,-30122.0,0.860
3,2023-04-01,Actual,193150.0,204900.0,-11750.0,0.943
4,2023-05-01,Actual,209624.0,212400.0,-2776.0,0.987
5,2023-06-01,Actual,205503.0,240000.0,-34497.0,0.856
6,2023-07-01,Actual,232778.0,248500.0,-15722.0,0.937
7,2023-08-01,Actual,246130.0,245300.0,830.0,1.003
8,2023-09-01,Actual,261139.0,248500.0,12639.0,1.051
9,2023-10-01,Actual,279982.0,275900.0,4082.0,1.015


## 7. Scenarios

Each scenario carries a stated assumption rather than an unexplained multiplier,
and the assumptions are sourced from the descriptive layer rather than invented.
The downside margin reduction is tied to the inventory finding that 273K units sit
in the 181+ day age bucket — clearing that stock means discounting it.

In [16]:
connection.execute("""
    SELECT scenario,
           ROUND(SUM(forecast_net_sales_usd), 0)  AS net_sales_usd,
           ROUND(SUM(forecast_net_profit_usd), 0) AS net_profit_usd,
           ROUND(MAX(assumed_margin_rate), 4)     AS assumed_margin,
           MAX(volume_multiplier)                 AS volume_multiplier,
           MAX(narrative)                         AS assumption
    FROM mart.forecast_scenarios GROUP BY 1 ORDER BY 2 DESC
""").fetchdf()

,scenario,net_sales_usd,net_profit_usd,assumed_margin,volume_multiplier,assumption
0,Upside,5664792.0,2965461.0,0.5235,1.1,Peak-season sell-through runs 10% ahead with a...
1,Base,5149811.0,2670125.0,0.5185,1.0,Current trend continues; no change to pricing ...
2,Downside,4634830.0,2310416.0,0.4985,0.9,Demand softens 10% and clearing aged stock cos...


## 8. The monthly scorecard

The artifact that actually circulates. Plan targets exist for net sales and units
only; everything else is tracked against prior month and last year, and the RAG
status is blank rather than green where there is no target to compare against.

In [17]:
connection.execute("""
    SELECT kpi_name, reporting_month,
           ROUND(actual_value, 2) AS actual,
           ROUND(mom_variance_rate, 4) AS mom,
           ROUND(yoy_variance_rate, 4) AS yoy,
           ROUND(plan_achievement_rate, 4) AS vs_plan,
           plan_rag_status
    FROM mart.commercial_scorecard ORDER BY kpi_order
""").fetchdf()

,kpi_name,reporting_month,actual,mom,yoy,vs_plan,plan_rag_status
0,Net Sales,2024-04-01,550157.88,0.2466,1.8483,1.5291,Green
1,Net Profit,2024-04-01,285251.26,0.2451,1.8492,NaN,No plan
2,Net Margin Rate,2024-04-01,0.52,-0.0012,0.0003,NaN,No plan
3,Units Sold,2024-04-01,12313.00,0.2264,1.8319,2.0399,Green
4,Orders,2024-04-01,8703.00,0.2220,1.8441,NaN,No plan
5,Active Customers,2024-04-01,7656.00,0.1657,1.5563,NaN,No plan
6,Average Order Value,2024-04-01,63.21,0.0201,0.0015,NaN,No plan
7,Item Return Rate,2024-04-01,0.11,0.0889,0.1629,NaN,No plan
8,Item Cancellation Rate,2024-04-01,0.15,0.0013,-0.0545,NaN,No plan


## 9. What this dataset cannot support

Four commercial concepts are deliberately **not** built here, because the data
cannot support them honestly:

| Concept | Blocking limitation |
|---|---|
| Price elasticity | No discount or promotion field; price is effectively fixed per product, so there is no price variation to regress against |
| Promotion effectiveness | No campaign table, no exposure flag, no control group |
| Marketing incrementality / ROAS | No marketing cost and no holdout |
| Multi-touch attribution | One traffic source per session, no touchpoint sequence per user |

Section 9 of [`docs/commercial_analytics.md`](../docs/commercial_analytics.md)
states the data and the method each one would require. Producing a coefficient
from data that cannot identify it would be worse than reporting the gap.

In [18]:
connection.close()
print("Warehouse connection closed.")

Warehouse connection closed.
